# Exploration des données

Examiner les données brutes avant toute transformation. Le dataset doit être placé dans `data/telco_customers.csv`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / 'app.py').exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

df = pd.read_csv(ROOT / "data/telco_customers.csv")
df.shape


In [ ]:
df.head()


In [ ]:
df.dtypes.to_frame("Type")


## Qualité des données

Les chaînes vides de TotalCharges deviennent des valeurs manquantes. Elles sont exclues explicitement du jeu utilisé par le script d’entraînement.


In [ ]:
numeric_total = pd.to_numeric(df["TotalCharges"], errors="coerce")
pd.Series({"rows": len(df), "missing_total_charges": int(numeric_total.isna().sum()), "duplicate_rows": int(df.duplicated().sum())})


In [ ]:
df["Churn"].value_counts().rename_axis("Churn").to_frame("Customers")


In [ ]:
pd.crosstab(df["Contract"], df["Churn"], normalize="index").round(3)


## Préparation des dix variables

Cette étape valide les caractéristiques et la cible. Aucun encodeur ni normaliseur n’est ajusté sur l’ensemble des données.


In [ ]:
from app import validate_data

def prepare_training_data(frame):
    if 'Churn' not in frame.columns:
        raise ValueError('The training dataset must include Churn.')
    frame = frame.copy()
    frame['TotalCharges'] = pd.to_numeric(frame['TotalCharges'], errors='coerce')
    frame = frame.dropna(subset=['TotalCharges'])
    target = frame['Churn'].astype('string').str.strip().map({'No': 0, 'Yes': 1, '0': 0, '1': 1})
    if target.isna().any() or target.nunique() != 2:
        raise ValueError('Churn must contain both No and Yes classes.')
    return validate_data(frame, max_rows=None), target.astype(int)

features, target = prepare_training_data(df)
features.describe(include="all").T
